<a href="https://colab.research.google.com/github/adrianhernr/Finance/blob/main/Top%20Momentum%20Nasdaq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TOP MOMENTUM STOCKS OF NASDAQ-100**

In [12]:
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime, timedelta
import io

def obtener_tickers_nasdaq100():
    """Descarga automáticamente los tickers actualizados del Nasdaq 100 desde Wikipedia."""
    url = 'https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies'

    # Cabecera simulando de forma idéntica a un navegador Chrome real en Windows 10
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9'
    }

    # Hacemos la petición
    respuesta = requests.get(url, headers=headers)

    # Envolvemos el texto HTML en StringIO para que pandas lo procese correctamente
    html_data = io.StringIO(respuesta.text)
    tablas = pd.read_html(html_data)

    # Extraemos la primera tabla de la página
    df_componentes = tablas[0]

    # Obtenemos los símbolos bursátiles de la columna 'Ticker'
    tickers = df_componentes['Ticker'].tolist()

    # Reemplazamos los puntos por guiones para que Yahoo Finance los reconozca
    tickers = [t.replace('.', '-') for t in tickers]
    return tickers

def calcular_top_momentum_nasdaq(limite_top=10):
    # 1. Obtener el universo del Nasdaq 100 de forma dinámica
    print("Obteniendo componentes actualizados del Nasdaq 100...")
    tickers = obtener_tickers_nasdaq100()

    # 2. Definir ventanas temporales (Estrategia institucional 12-1 meses)
    fecha_hoy = datetime.today()
    fecha_inicio = fecha_hoy - timedelta(days=365)
    fecha_corte_reciente = fecha_hoy - timedelta(days=30)

    print(f"Descargando datos históricos para {len(tickers)} empresas...")
    # CORRECCIÓN DE LA DESCARGA:
    datos = yf.download(
        tickers,
        start=fecha_inicio,
        end=fecha_hoy,
        progress=False,
        auto_adjust=False,  # Recupera la columna 'Adj Close'
        group_by='column'   # Mantiene el MultiIndex con los tipos de precio arriba
    )['Adj Close']

    resultados_momentum = {}

    # 3. Calcular la fuerza del momentum para cada acción
    for ticker in tickers:
        if ticker in datos.columns:
            serie_precios = datos[ticker].dropna()

            # Filtrar el rango excluyendo los últimos 30 días
            precios_filtrados = serie_precios.loc[:fecha_corte_reciente.strftime('%Y-%m-%d')]

            if len(precios_filtrados) > 20 and len(serie_precios) > 20:
                precio_inicial = precios_filtrados.iloc[0]
                precio_intermedio = precios_filtrados.iloc[-1]
                precio_actual = serie_precios.iloc[-1]

                # Condición de filtro extra: Solo acciones en tendencia alcista estructural
                if precio_actual > precio_inicial:
                    # Fórmula del Momentum 12-1
                    retorno = ((precio_intermedio - precio_inicial) / precio_inicial) * 100
                    resultados_momentum[ticker] = retorno

    # 4. Procesar y ordenar el Top definitivo
    df_momentum = pd.DataFrame(list(resultados_momentum.items()), columns=['Ticker', 'Rendimiento_12_1_%'])
    df_momentum = df_momentum.sort_values(by='Rendimiento_12_1_%', ascending=False).reset_index(drop=True)

    return df_momentum.head(limite_top)

if __name__ == "__main__":
    top_nasdaq = calcular_top_momentum_nasdaq(limite_top=15)

    print("\n" + "="*50)
    print("   TOP 15 MOMENTUM (12-1M) - COMPONENTES NASDAQ 100   ")
    print("="*50)
    print(top_nasdaq.to_string(index=False, formatters={'Rendimiento_12_1_%': '{:,.2f}%'.format}))


Obteniendo componentes actualizados del Nasdaq 100...
Descargando datos históricos para 103 empresas...

   TOP 15 MOMENTUM (12-1M) - COMPONENTES NASDAQ 100   
Ticker Rendimiento_12_1_%
  SNDK          4,634.90%
    MU            847.20%
   WDC            662.48%
  LITE            627.80%
  INTC            541.52%
   STX            487.70%
  NBIS            321.05%
  LRCX            314.83%
   TER            298.80%
  AMAT            264.15%
  MRVL            239.13%
  ALAB            215.12%
   AMD            206.78%
  KLAC            204.77%
  ASML            166.70%
